# `json2vec` Hello World

This notebook trains a tiny model from an in-memory synthetic dataset.

In [ ]:
import lightning.pytorch as lit
import polars as pl
import torch
from rich.pretty import pprint

import json2vec as j2v

In [ ]:
records = pl.DataFrame(
    {
        "color": ["red", "orange", "yellow", "blue", "green", "purple"],
        "label": ["warm", "warm", "warm", "cool", "cool", "cool"],
    }
)

In [ ]:
model = j2v.Model.from_schema(
    j2v.Column("color", j2v.Category, max_vocab_size=16),
    j2v.Column("label", j2v.Category, target=True, max_vocab_size=8, topk=[2]),
    d_model=16,
    n_layers=1,
    n_heads=4,
    batch_size=4,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)

datamodule = j2v.PolarsDataModule.from_model(
    model,
    train=records,
    validate=records,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    observation_buffer_size=32,
    sample_rate=1.0,
)


In [ ]:
trainer = lit.Trainer(
    accelerator="cpu",
    max_epochs=20,
    logger=False,
    enable_progress_bar=False,
    enable_model_summary=False,
    enable_checkpointing=False,
)

trainer.fit(model=model, datamodule=datamodule)


In [ ]:
batch = [[{"color": "red"}], [{"color": "blue"}]]

In [ ]:
pprint(model.predict(batch))

In [ ]:
model.set(j2v.where("name") == "record", embed=True)

pprint(model.embed(batch))

In [ ]:
model.plot(detail=True)